In [ ]:
import pandas as pd
import unicodedata

Importation des données de toutes les élections pour élections municipales

In [3]:
all_elections = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/candidats_results.txt", sep=";")

/tmp/ipykernel_11314/4016539827.py:1: DtypeWarning: Columns (2,4,6,7,8,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  all_elections = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/candidats_results.txt", sep=";")


## Etape initiale : constituer une liste des nuances de tous les candidats

In [ ]:
def enlever_accents(text):
    # Normalise en forme NFD (décompose les accents)
    text_normalized = unicodedata.normalize('NFD', text)
    # Garde seulement les caractères ASCII (supprime les accents)
    text_ascii = text_normalized.encode('ascii', 'ignore').decode('utf-8')
    return text_ascii

In [ ]:
all_elections.head()
liste_candidats_nuance = all_elections['Nom', 'Prénom', 'Nuance']

,id_election,id_brut_miom,Code du département,Code de la commune,Code du b.vote,N°Panneau,Libellé Abrégé Liste,Libellé Etendu Liste,Nom Tête de Liste,Voix,% Voix/Ins,% Voix/Exp,Sexe,Nom,Prénom,Nuance,Binôme,Liste
0,2019_euro_t1,01001_0001,01,1,0001,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,13.0,2.16,4.14,NaN,NaN,NaN,NaN,NaN,NaN
1,2019_euro_t1,01002_0001,01,2,0001,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,6.0,2.86,4.44,NaN,NaN,NaN,NaN,NaN,NaN
2,2019_euro_t1,01004_0001,01,4,0001,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,39.0,3.71,8.23,NaN,NaN,NaN,NaN,NaN,NaN
3,2019_euro_t1,01004_0002,01,4,0002,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,42.0,3.80,7.78,NaN,NaN,NaN,NaN,NaN,NaN
4,2019_euro_t1,01004_0003,01,4,0003,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,31.0,2.93,5.63,NaN,NaN,NaN,NaN,NaN,NaN


In [93]:
elec_binome = all_elections[all_elections["Binôme"].notna()]
liste_elec_binome = elec_binome['id_election'].unique()
election_dep = all_elections[all_elections['id_election'].isin(liste_elec_binome)]
election_dep.drop_duplicates(subset=["id_election", "id_brut_miom", "Code du département","Nuance", "Binôme"], inplace=True)

print(election_dep['id_election'].unique())
election_dep = election_dep[["id_election", "id_brut_miom", "Code du département","Nuance", "Binôme"]]
election_dep[["candidat1", "candidat2"]] = election_dep["Binôme"].str.split(" et ", expand=True)

# Enlever les titres "M" ou "Mme" au début
election_dep["candidat1"] = election_dep["candidat1"].str.replace(r"^(M\.|Mme)\s+", "", regex=True)
election_dep["candidat2"] = election_dep["candidat2"].str.replace(r"^(M\.|Mme)\s+", "", regex=True)

election_dep[[ "nom1", "prenom1"]] = election_dep["candidat1"].str.split(" ", n=1, expand=True)
election_dep["prenom1"] = election_dep["prenom1"].str.lower()
# Séparer prénom et nom pour person2
election_dep[[ "nom2","prenom2"]] = election_dep["candidat2"].str.split(" ", n=1, expand=True)
election_dep["prenom2"] = election_dep["prenom2"].str.lower()

election_dep.head()
election_dep_long = pd.wide_to_long(election_dep, 
                          stubnames=["prenom", "nom"], 
                          i=["id_election", "id_brut_miom", "Code du département","Nuance", "Binôme"], 
                          j="num_binôme",   # nouveau suffixe qui indiquera 1 ou 2
                          sep="")    # pas de séparateur entre stubname et numéro

# Réinitialiser l'index pour obtenir un DataFrame classique
election_dep_long = election_dep_long.reset_index()

/tmp/ipykernel_11314/2525613488.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  election_dep.drop_duplicates(subset=["id_election", "id_brut_miom", "Code du département","Nuance", "Binôme"], inplace=True)


['2021_dpmt_t2' '2021_dpmt_t1' '2015_dpmt_t1' '2015_dpmt_t2']


In [94]:
election_dep_long.head()

,id_election,id_brut_miom,Code du département,Nuance,Binôme,num_binôme,candidat1,candidat2,prenom,nom
0,2021_dpmt_t2,01001_0001,01,BC-UCD,Mme CHMARA Patricia et M. MATHIAS Patrick,1,CHMARA Patricia,MATHIAS Patrick,patricia,CHMARA
1,2021_dpmt_t2,01001_0001,01,BC-UCD,Mme CHMARA Patricia et M. MATHIAS Patrick,2,CHMARA Patricia,MATHIAS Patrick,patrick,MATHIAS
2,2021_dpmt_t2,01002_0001,01,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,1,PEREYRON Fabrice,RAY Marie-Céline,fabrice,PEREYRON
3,2021_dpmt_t2,01002_0001,01,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,2,PEREYRON Fabrice,RAY Marie-Céline,marie-céline,RAY
4,2021_dpmt_t2,01004_0001,01,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,1,PEREYRON Fabrice,RAY Marie-Céline,fabrice,PEREYRON


In [84]:
election_dep.head()

,id_election,id_brut_miom,Code du département,Nuance,Binôme,candidat1,candidat2
4005041,2021_dpmt_t2,01001_0001,01,BC-UCD,Mme CHMARA Patricia et M. MATHIAS Patrick,CHMARA Patricia,MATHIAS Patrick
4005042,2021_dpmt_t2,01002_0001,01,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,PEREYRON Fabrice,RAY Marie-Céline
4005043,2021_dpmt_t2,01004_0001,01,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,PEREYRON Fabrice,RAY Marie-Céline
4005044,2021_dpmt_t2,01004_0002,01,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,PEREYRON Fabrice,RAY Marie-Céline
4005045,2021_dpmt_t2,01004_0003,01,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,PEREYRON Fabrice,RAY Marie-Céline


In [ ]:
donnees_restreintes_muni = all_elections[all_elections["id_election"].str.contains("muni", case=False, na=False)]
donnees_restreintes_muni.head()

,id_election,id_brut_miom,Code du département,Code de la commune,Code du b.vote,N°Panneau,Libellé Abrégé Liste,Libellé Etendu Liste,Nom Tête de Liste,Voix,% Voix/Ins,% Voix/Exp,Sexe,Nom,Prénom,Nuance,Binôme,Liste
4005041,2021_dpmt_t2,01001_0001,01,1,0001,1.0,NaN,NaN,NaN,112.0,18.01,58.03,NaN,NaN,NaN,BC-UCD,Mme CHMARA Patricia et M. MATHIAS Patrick,NaN
4005042,2021_dpmt_t2,01002_0001,01,2,0001,1.0,NaN,NaN,NaN,36.0,17.31,39.13,NaN,NaN,NaN,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,NaN
4005043,2021_dpmt_t2,01004_0001,01,4,0001,1.0,NaN,NaN,NaN,90.0,8.47,38.79,NaN,NaN,NaN,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,NaN
4005044,2021_dpmt_t2,01004_0002,01,4,0002,1.0,NaN,NaN,NaN,92.0,8.39,32.86,NaN,NaN,NaN,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,NaN
4005045,2021_dpmt_t2,01004_0003,01,4,0003,1.0,NaN,NaN,NaN,121.0,10.64,36.89,NaN,NaN,NaN,BC-UG,M. PEREYRON Fabrice et Mme RAY Marie-Céline,NaN


In [31]:
donnees_restreintes_muni[donnees_restreintes_muni['Nom'] == "CHABRY"]

,id_election,id_brut_miom,Code du département,Code de la commune,Code du b.vote,N°Panneau,Libellé Abrégé Liste,Libellé Etendu Liste,Nom Tête de Liste,Voix,% Voix/Ins,% Voix/Exp,Sexe,Nom,Prénom,Nuance,Binôme,Liste
4268119,2020_muni_t2,03132_0001,03,132,0001,2.0,NaN,NaN,NaN,0.0,0.00,0.0,M,CHABRY,Bruno,NC,NaN,NaN
4352421,2020_muni_t2,24488_0001,24,488,1,38.0,NaN,NaN,NaN,65.0,18.36,24.9,M,CHABRY,Michel,NC,NaN,NaN
10028386,2020_muni_t1,03132_0001,03,132,0001,2.0,NaN,NaN,NaN,136.0,29.37,NaN,M,CHABRY,Bruno,NC,NaN,NaN
10097351,2020_muni_t1,03299_0001,03,299,0001,3.0,NaN,NaN,NaN,73.0,45.63,NaN,M,CHABRY,Jérôme,NC,NaN,NaN
10119614,2020_muni_t1,42070_0001,42,70,0001,3.0,NaN,NaN,NaN,305.0,50.75,NaN,M,CHABRY,Julien,NC,NaN,NaN
10423176,2020_muni_t1,07306_0001,07,306,0001,11.0,NaN,NaN,NaN,128.0,58.45,NaN,M,CHABRY,Christophe,NC,NaN,NaN
10570145,2020_muni_t1,24488_0001,24,488,1,38.0,NaN,NaN,NaN,56.0,15.77,NaN,M,CHABRY,Michel,NC,NaN,NaN
26423281,2014_muni_t1,03132_0001,3,132,1,4.0,NaN,NaN,NaN,241.0,45.56,NaN,NaN,CHABRY,Geneviève,NC,NaN,NaN
26425704,2014_muni_t1,03299_0001,3,299,1,8.0,NaN,NaN,NaN,129.0,66.49,NaN,NaN,CHABRY,Jérôme,NC,NaN,NaN
26541643,2014_muni_t1,24488_0001,24,488,1,19.0,NaN,NaN,NaN,78.0,20.58,NaN,NaN,CHABRY,Michel,NC,NaN,NaN


In [5]:
# ---- Étape 1 : calcul des voix totales par élection et ville ----
voix_totales = (
    donnees_restreintes_muni
    .assign(ident_election_ville=lambda d: d["id_brut_miom"].str[:5])
    .groupby(["id_election", "ident_election_ville"], as_index=False)
    .agg(voix_total_ville_elec=("Voix", "sum"))
)

In [6]:

# ---- Étape 2 : calcul des voix par candidat ----
donnees_muni_long = (
    donnees_restreintes_muni
    .assign(ident_election_ville=lambda d: d["id_brut_miom"].str[:5])
    .groupby(["id_election", "ident_election_ville", "Nom", "Prénom", "Nuance"], as_index=False)
    .agg(voix_totales_candidat=("Voix", "sum"))
    .merge(voix_totales, on=["id_election", "ident_election_ville"], how="left")
    .assign(voix_pct=lambda d: d["voix_totales_candidat"] / d["voix_total_ville_elec"] * 100)
    .sort_values(["id_election", "ident_election_ville", "voix_pct"], ascending=[True, True, False])
)

# ---- Étape 3 : classement des deux premiers ----
donnees_muni_wide = (
    donnees_muni_long
    .assign(rang=lambda d: d.groupby(["id_election", "ident_election_ville"]).cumcount() + 1)
    .query("rang <= 2")
)

# ---- Étape 4 : passage en format large ----
donnees_muni_wide = (
    donnees_muni_wide
    .pivot(index=["id_election", "ident_election_ville"],
           columns="rang",
           values=["Prénom", "Nom", "Nuance","voix_pct"])
)

# Aplatir les noms de colonnes hiérarchiques comme "rang1_Prénom"
donnees_muni_wide.columns = [
    f"rang{r}_{v}" for v, r in donnees_muni_wide.columns
]
donnees_muni_wide = donnees_muni_wide.reset_index()

# ---- Étape 5 : extraire les variables election / tour ----
donnees_muni_wide = (
    donnees_muni_wide
    .assign(
        election=lambda d: d["id_election"].str[:9],
        tour=lambda d: d["id_election"].str[10:12])
    .groupby(["election", "ident_election_ville"], group_keys=False)
    .filter(lambda g: not ("t1" in g["tour"].values and "t2" in g["tour"].values and g["tour"].eq("t1").any()))
)


In [7]:
donnees_muni_wide.head()

,id_election,ident_election_ville,rang1_Prénom,rang2_Prénom,rang1_Nom,rang2_Nom,rang1_Nuance,rang2_Nuance,rang1_voix_pct,rang2_voix_pct,election,tour
1,2008_muni_t1,01014,Liliane,Michel,MAISSIAT,GAUTHIER,LMAJ,LDVD,78.73701,21.26299,2008_muni,t1
2,2008_muni_t1,01033,Régis,Guy,PETIT,LARMANJAT,LMAJ,LSOC,58.224883,41.775117,2008_muni,t1
5,2008_muni_t1,01053,Jean-François,Christophe,DEBAT,FEILLENS,LSOC,LMAJ,55.371775,34.802731,2008_muni,t1
7,2008_muni_t1,01142,Bernard,Bernard,SIMPLEX,LOBIETTI,LDVD,LMAJ,58.760951,41.239049,2008_muni,t1
8,2008_muni_t1,01143,Etienne,NaN,BLANC,NaN,LMAJ,NaN,100.0,NaN,2008_muni,t1


In [8]:
donnees_muni_long[donnees_muni_long['Nom'] == "Chabry"]

,id_election,ident_election_ville,Nom,Prénom,Nuance,voix_totales_candidat,voix_total_ville_elec,voix_pct


In [9]:
donnees_muni_long.head()

,id_election,ident_election_ville,Nom,Prénom,Nuance,voix_totales_candidat,voix_total_ville_elec,voix_pct
1,2008_muni_t1,01004,EXPOSITO,Josiane,LUG,1341.0,4272.0,31.390449
0,2008_muni_t1,01004,CASTELLANO,Sandrine,LDVD,1327.0,4272.0,31.062734
2,2008_muni_t1,01004,PAVIER,Bernard,LGC,820.0,4272.0,19.194757
3,2008_muni_t1,01004,SASSO,Jean-Marc,LDVD,784.0,4272.0,18.352060
5,2008_muni_t1,01014,MAISSIAT,Liliane,LMAJ,985.0,1251.0,78.737010


In [ ]:

#on prépare la suite en créant un fichier listant les têtes de liste des municipales et leur couleur politique
liste_candidats_nuance = (
                    donnees_muni_long
                    .assign(annee = lambda d : d["id_election"].str[0:4])
                    .assign(dep = lambda d : d["ident_election_ville"].str[0:2])
                    .loc[:, ["annee","dep", "Nom", "Prénom", "Nuance"]]
)
liste_candidats_nuance['Prénom'] = liste_candidats_nuance['Prénom'].str.lower().apply(enlever_accents)
liste_candidats_nuance.head()


,annee,dep,Nom,Prénom,Nuance
1,2008,01,EXPOSITO,josiane,LUG
0,2008,01,CASTELLANO,sandrine,LDVD
2,2008,01,PAVIER,bernard,LGC
3,2008,01,SASSO,jean-marc,LDVD
5,2008,01,MAISSIAT,liliane,LMAJ


Importation des données des intercommunalités

In [60]:
grp_2025 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/liste-des-groupements-france-entiere-20250127.csv", encoding="latin-1", sep = ";")
grp_2019 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/Liste_des_groupements_en_2019.csv", encoding="latin-1", sep = ";")
grp_2013 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/Liste_des_groupements_en_2013.csv", encoding="latin-1", sep = ";")
grp_2013['annee'] = "2008"
grp_2019['annee'] = "2014"
grp_2025['annee'] = "2020"

grp_2013['Prénom'] = grp_2013['Prénom Président'].str.lower()
grp_2019['Prénom'] = grp_2019['Prénom Président'].str.lower()
grp_2025['Prénom'] = grp_2025['Prénom Président'].str.lower()


grp_2013['dep'] = grp_2013["Département siège"].str[:2]
grp_2019['dep'] = grp_2019["Département siège"].str[:2]
#grp_2025['dep'] = grp_2025["Département siège"].str[:2]



In [61]:
liste_candidats_nuance[liste_candidats_nuance['Nom'] == "DEBAT"]

,annee,dep,Nom,Prénom,Nuance
14,2008,01,DEBAT,jean-francois,LSOC
5076,2008,75,DEBAT,vincent,LEXG
7148,2008,94,DEBAT,martine,LEXG
10983,2014,01,DEBAT,jean francois,LUG
91752,2014,18,DEBAT,jean-claude,NC
161798,2014,32,DEBAT,alain,NC
164368,2014,32,DEBAT,jean-michel,NC
165092,2014,32,DEBAT,andre,NC
169993,2014,33,DEBAT,sylvie,NC
326474,2014,65,DEBAT,christine,NC


ATTENTION : penser à rajouter l'année, pour les candidats changeant de parti, sinon résultats bizarres

In [62]:
grp_2019.shape

(11226, 23)

In [63]:
grp_2019_avec_nuance = pd.merge(
    grp_2019,
    liste_candidats_nuance,
    how="left",  # ou "inner", "right", "outer" selon ton besoin
    left_on=["Prénom", "Nom Président", "annee", "dep"],
    right_on=["Prénom", "Nom", "annee", "dep"]
)

In [64]:
print(grp_2019_avec_nuance.shape)
print(grp_2019_avec_nuance['Nuance'].isna().sum())

(11925, 25)
5142


In [71]:
liste_candidats_nuance[
    (liste_candidats_nuance['Prénom'] == "philippe") &
    (liste_candidats_nuance['Nom'] == "GUILLOT-VIGNOT")
]

,annee,dep,Nom,Prénom,Nuance


In [ ]:
non_matched = merge_4[merge_4['score'].isna()].drop(columns=['score'])
merge_3 = non_matched.merge(df2, on=["prenom", "nom", "ville"], how="left", suffixes=('', '_df2'))


In [68]:
grp_2019_avec_nuance.head(20)

,Région siège,Département siège,Arrondissement siège,Commune siège,N° SIREN,Nom du groupement,Nature juridique,Syndicat à la carte,Groupement interdépartemental,Date de création,...,Nombre de compétences exercées,Mode de financement,Civilité Président,Prénom Président,Nom Président,annee,Prénom,dep,Nom,Nuance
0,84 - Auvergne-Rhône-Alpes,01 - Ain,4 - Nantua,210102836 - Oyonnax,200042935,Haut - Bugey Agglomération,CA,0,0,01/01/2014,...,35,FPU,M.,Jean,DEGUERRY,2014,jean,01,DEGUERRY,LDVD
1,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100533 - Bourg-en-Bresse,200071751,CA du Bassin de Bourg-en-Bresse,CA,0,0,16/12/2016,...,41,FPU,M.,JEAN FRANCOIS,DEBAT,2014,jean francois,01,DEBAT,LUG
2,84 - Auvergne-Rhône-Alpes,01 - Ain,3 - Gex,210101739 - Gex,240100750,CA du Pays de Gex,CA,0,0,31/05/1995,...,35,FPU,M.,Christophe,BOUVIER,2014,christophe,01,BOUVIER,LDVD
3,84 - Auvergne-Rhône-Alpes,01 - Ain,4 - Nantua,210101994 - Jujurieux,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,0,0,25/11/2011,...,18,FPU,M.,Thierry,DUPUIS,2014,thierry,01,DUPUIS,LDVG
4,84 - Auvergne-Rhône-Alpes,01 - Ain,1 - Belley,210100343 - Belley,200040350,CC Bugey Sud,CC,0,0,01/01/2014,...,22,FPU,M.,RENE,VUILLEROD,2014,rene,01,VUILLEROD,NC
5,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210104279 - Trévoux,200042497,CC Dombes Saône Vallée,CC,0,0,01/01/2014,...,28,FPU,M.,Bernard,GRISON,2014,bernard,01,GRISON,LDVD
6,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100939 - Châtillon-sur-Chalaronne,200069193,CC de la Dombes,CC,0,0,01/12/2016,...,21,FPU,M.,MICHEL,GIRER,2014,michel,01,NaN,NaN
7,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210102588 - Montceaux,200070118,CC Val de Saône Centre,CC,0,0,06/12/2016,...,23,FPU,M.,Jean Claude,DESCHIZEAUX,2014,jean claude,01,NaN,NaN
8,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210103065 - Pont-de-Veyle,200070555,CC de la Veyle,CC,0,0,08/12/2016,...,22,FPU,M.,CHRISTOPHE,GREFFET,2014,christophe,01,GREFFET,NC
9,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100269 - Bâgé-le-Châtel,200071371,CC Bresse et Saône,CC,0,0,15/12/2016,...,22,FPU,M.,GUY,BILLOUDET,2014,guy,01,BILLOUDET,LDVD


In [23]:
liste_candidats_nuance[(liste_candidats_nuance["Prénom"] == "Jean") & (liste_candidats_nuance["Nom"] == "Chabry")]

,annee,dep,Nom,Prénom,Nuance


In [24]:
grp_2013_avec_nuance.head()

,Région siège,Département siège,Arrondissement siège,Commune siège,N° SIREN,Nom du groupement,Nature juridique,Syndicat à la carte,Groupement interdépartemental,Date de création,...,Mode de financement,Civilité Président,Prénom Président,Nom Président,annee,dep_x,dep_y,Nom,Prénom,Nuance
0,82 - Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100533 - Bourg-en-Bresse,240100628,CA Bourg en Bresse Agglomération,CA,0,0,28/11/2000,...,FPU,M.,Michel,FONTAINE,2008,01,ZD,FONTAINE,Michel,LMAJ
1,82 - Rhône-Alpes,01 - Ain,4 - Nantua,210101994 - Jujurieux,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,0,0,25/11/2011,...,FPU,M,Jean,CHABRY,2008,01,NaN,NaN,NaN,NaN
2,82 - Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100939 - Châtillon-sur-Chalaronne,200035210,CC Chalaronne Centre,CC,0,0,01/01/2013,...,ASZSE,M,Patrice,MORANDAS,2008,01,NaN,NaN,NaN,NaN
3,82 - Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210102661 - Montrevel-en-Bresse,240100156,CC de Montrevel - en - Bresse,CC,0,0,01/01/2002,...,ASZSE,M.,Jean Pierre,ROCHE,2008,01,NaN,NaN,NaN,NaN
4,82 - Rhône-Alpes,01 - Ain,4 - Nantua,210102836 - Oyonnax,240100172,CC d'Oyonnax,CC,0,0,19/06/2000,...,FPU,M.,Alexandre,TACHDJIAN,2008,01,NaN,NaN,NaN,NaN
